# 🚀 مشروع الذكاء الاصطناعي للتنبؤ بأسعار وعوائد العملات المشفرة
## Crypto Price Prediction AI: Production-Grade, Leakage-Free Forecasting System

مرحباً بك في الدفتر التطبيقي الشامل لنظام **Crypto Price Prediction AI**.
تم تصميم وبناء هذا المشروع وفق أرقى المعايير الهندسية والمالية للتنبؤ بأسعار وعوائد العملات المشفرة:
- **Bitcoin (BTCUSDT)**
- **Ethereum (ETHUSDT)**
- **Solana (SOLUSDT)**

### المحاور الأساسية لهذا الدفتر:
1. **البيئة والإعدادات (Setup & Dependencies):** استيراد الحزم وتهيئة بيئة العمل.
2. **استجلاب البيانات والتحقق (Data Ingestion & Cleaning):** جلب البيانات الحية من Binance أو العينات غير المتصلة وفحص سلامتها.
3. **التحليل الاستكشافي واختبار الاستقرارية (EDA & Stationarity):** فحص توزيع العوائد واختبار ديكـي-فولر (ADF Test).
4. **هندسة الخصائص السببية (Strictly Causal Feature Engineering):** بناء المؤشرات الفنية دون أي تسريب مستقبلي.
5. **رصد النماذج الفنية والشموع (Pattern Recognition):** الكشف عن المطارق، الشهب، الابتلاع، ونماذج القمم والقيعان المزدوجة.
6. **تدقيق منع تسريب البيانات (Data Leakage & Perturbation Test):** اختبار التشويش الرياضي لضمان خلو النظام من التحيز.
7. **التقسيم الزمني والتطهير (Chronological Splitting & Purging Gap):** عزل فترات التدريب والتحقق والاختبار بفجوة التطهير.
8. **تدريب ومقارنة النماذج (Model Comparison & Baselines):** مقارنة الخطوط المرجعية مع Ridge, Random Forest, XGBoost, LightGBM, CatBoost.
9. **التنبؤ الحي بالأفق الزمني المرن (Dynamic Arbitrary Forecasting):** التنبؤ بأي ساعة مستقبلية وعرض المخطط التفاعلي ومخروط عدم اليقين.
10. **تحسين المعاملات الفائقة (Optuna Bayesian Optimization):** الضبط البايزي عبر التحقق المتقاطع بالنافذة المتنقلة.


## 1. تهيئة بيئة العمل والاستيراد (Environment Setup & Imports)
نقوم أولاً بالتحقق من مسار الكود المصدري `src` وإضافته إلى مسار النظام `sys.path`، ثم استيراد كافة الوحدات المطورة داخل الحزمة.


In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ضبط مسار الحزمة البرمجية src
current_dir = Path.cwd()
src_path = current_dir.parent / "src" if (current_dir.parent / "src").exists() else current_dir / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# استيراد وحدات النظام الأساسية
from crypto_prediction.data.collector import DataCollector
from crypto_prediction.data.validator import DataValidator
from crypto_prediction.data.cleaner import DataCleaner
from crypto_prediction.features.engineer import FeatureEngineer
from crypto_prediction.features.patterns import PatternRecognizer
from crypto_prediction.validation.leakage_guard import LeakageGuard
from crypto_prediction.validation.splitter import ChronologicalSplitter
from crypto_prediction.pipeline.evaluator import ModelEvaluator
from crypto_prediction.pipeline.trainer import PipelineTrainer
from crypto_prediction.pipeline.optimizer import HyperparameterOptimizer
from crypto_prediction.utils.chart_plotter import AdvancedChartVisualizer

print("✅ تم استيراد حزمة Crypto Price Prediction AI بنجاح!")


## 2. جمع البيانات والتحقق (Data Ingestion & Fallback)
يدعم النظام سحب البيانات الحية عبر Binance Public REST API مباشرة، وفي حال عدم توفر اتصال بالإنترنت، ينتقل تلقائياً للعمل دون اتصال (**Offline Fallback Mode**) عبر البيانات المخزنة محلياً.


In [ ]:
symbol = "BTCUSDT"
interval = "1h"
total_bars = 1000

collector = DataCollector()
raw_df = collector.fetch_ohlcv(symbol=symbol, interval=interval, total_bars=total_bars)

print(f"تم جلب {len(raw_df)} شمعة للرمز {symbol} بنجاح.")
display(raw_df.head(5))


## 3. التحقق من سلامة البيانات وتنظيفها (Validation & Cleaning)
نقوم بفحص البيانات للكشف عن أي قيم سالبة، فجوات زمنية، أو تناقضات شمعية ($High \ge Low$)، ثم تنظيفها وترتيبها زمنياً تصاعدياً مع تحويل التوقيت إلى DatetimeIndex موحد.


In [ ]:
# 1. التحقق الآلي
validator = DataValidator(expected_interval=interval)
val_report = validator.validate(raw_df)
print("تقرير التحقق من سلامة البيانات:")
print(f"- صالحة تماماً: {val_report.is_valid}")
print(f"- إجمالي الصفوف: {val_report.row_count}")
print(f"- عدد الفجوات الزمنية المكتشفة: {val_report.missing_bars_count}")

# 2. تنظيف البيانات وتوحيد المؤشر الزمني
cleaner = DataCleaner()
clean_df = cleaner.clean(raw_df)
print(f"\nتم تنظيف البيانات بنجاح: {len(clean_df)} شمعة زمنية نظيفة.")
display(clean_df.tail(3))


## 4. التحليل الاستكشافي واختبار الاستقرارية (EDA & Stationarity)
في السلاسل الزمنية المالية، لا يمكن تدريب النماذج الإحصائية على الأسعار المطلقة لأنها غير مستقرة إحصائياً (Non-stationary).
نقوم هنا بحساب العوائد اللحظية اللوغاريتمية وفحص استقرارها عبر اختبار **Augmented Dickey-Fuller (ADF)**.


In [ ]:
clean_df["log_return"] = np.log(clean_df["close"] / clean_df["close"].shift(1))

# اختبار ديكي-فولر على السعر مقابل العائد
try:
    from statsmodels.tsa.stattools import adfuller
    adf_price = adfuller(clean_df["close"].dropna())
    adf_return = adfuller(clean_df["log_return"].dropna())
    print(f"اختبار ADF لسعر الإغلاق: p-value = {adf_price[1]:.4f} (غير مستقر إذا كانت > 0.05)")
    print(f"اختبار ADF للعائد اللوغاريتمي: p-value = {adf_return[1]:.4e} (مستقر تماماً)")
except ImportError:
    print("تنبيه: مكتبة statsmodels غير مثبتة. لتشغيل اختبار ADF يرجى تثبيتها: pip install statsmodels")
    print(f"تباين السعر الخام: {clean_df['close'].var():.2f} (غير مستقر)")
    print(f"تباين العائد اللوغاريتمي: {clean_df['log_return'].var():.6f} (مستقر)")

# رسم السعر وتوزيع العوائد
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(clean_df.index, clean_df["close"], color="#2980b9", lw=1.5, label="سعر البيتكوين (USDT)")
ax1.set_title("حركة السعر التاريخية للبيتكوين", fontsize=12, fontweight="bold")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(clean_df.index, clean_df["log_return"], color="#27ae60", lw=0.8, alpha=0.8, label="العوائد اللوغاريتمية (1h)")
ax2.axhline(0, color="black", lw=0.8, ls="--")
ax2.set_title("العوائد اللحظية المستقرة (Stationary Series)", fontsize=12, fontweight="bold")
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. هندسة الخصائص السببية (Strictly Causal Feature Engineering)
نقوم بتوليد كافة المؤشرات الفنية (RSI, MACD, Bollinger Bands, Moving Averages, Volatility) مع الالتزام الصارم بالسببية الزمنية:
- نوافذ متدحرجة غير متمركزة فقط (`center=False`).
- عدم استخدام أي إزاحات مستقبلية أو ملء خلفي (`bfill`).
- محاذاة الهدف $y_{t} = (Close_{t+h} - Close_{t}) / Close_{t}$ بصورة منفصلة تماماً.


In [ ]:
forecast_horizon = 1  # التنبؤ بساعة واحدة للأمام

engineer = FeatureEngineer(
    return_lags=[1, 2, 3, 6, 12, 24],
    rolling_windows=[6, 12, 24, 72, 168],
    rsi_periods=[7, 14, 21],
    forecast_horizons=[1, 4, 12, 24],
    primary_horizon=forecast_horizon,
    include_patterns=True,
)

X, y, meta = engineer.prepare_dataset(clean_df, target_horizon=forecast_horizon)

print(f"تم توليد مصفوفة الخصائص: {X.shape[0]} صف و {X.shape[1]} خاصية سببية.")
print(f"الهدف المستهدف: {y.name} بمعدل {y.mean():.6f} وانحراف معياري {y.std():.6f}")
display(X.tail(3))


## 6. محرك رصد النماذج الفنية والشموع اليابانية (Pattern Recognition Engine)
يتضمن النظام محركاً متقدماً للكشف السببي عن أهم النماذج السعرية الكلاسيكية:
1. **المطارق (Hammers / Pinbars):** إشارات انعكاس شرائي ذات ذيل سفلي طويل.
2. **الشهب (Shooting Stars):** إشارات انعكاس بيعي ذات ذيل علوي طويل.
3. **الابتلاع (Engulfing):** شمعات ابتلاعية شرائية وبيعية.
4. **الدعم والمقاومة المتأرجحة (Dynamic Support & Resistance):** مستويات سعرية ديناميكية عبر قمم وقيعان النوافذ.
5. **النماذج الهندسية:** القمم المزدوجة (M-tops) والقيعان المزدوجة (W-bottoms).


In [ ]:
recognizer = PatternRecognizer()
detected_patterns = recognizer.scan_patterns(clean_df, recent_bars=100)

print(f"عدد النماذج الفنية المكتشفة في آخر 100 شمعة: {len(detected_patterns)}")
for pat in detected_patterns[-5:]:
    print(f"- {pat.name} ({pat.direction}) عند الشمعة {pat.timestamp} بقوة ثقة {pat.confidence*100:.0f}%")

# استعراض مستويات الدعم والمقاومة الحالية
sup, res = recognizer.compute_dynamic_support_resistance(
    clean_df["high"], clean_df["low"], clean_df["close"], window=48
)
print(f"\nمستوى الدعم الحالي (48h): ${sup.iloc[-1]:,.2f}")
print(f"مستوى المقاومة الحالي (48h): ${res.iloc[-1]:,.2f}")


## 7. تدقيق منع تسريب البيانات واختبار التشويش (Leakage Guard & Perturbation)
لضمان عدم وجود أي تسريب مستقبلي، نقوم بتنفيذ **اختبار التشويش المستقبلي (Future Perturbation Test)**:
نقوم بخلط ومضاعفة أسعار المستقبل بعد نقطة زمنية $T_{cutoff}$، ونعيد حساب الخصائص. إذا تغيرت أي خاصية في الماضي ولو بجزء من المليار، يفشل الاختبار فوراً.


In [ ]:
guard = LeakageGuard()
pert_passed, max_diff, msg = guard.run_perturbation_test(
    df_raw=clean_df,
    feature_engineer=engineer,
    cutoff_idx=int(len(clean_df) * 0.70),
)

print(f"نتيجة اختبار التشويش المستقبلي: {'ناجح تماماً ✅' if pert_passed else 'فاشل ❌'}")
print(f"أقصى فرق تم رصده في بيانات الماضي: {max_diff:.2e} (صفر مطلق)")
print(f"الرسالة التفصيلية: {msg}")


## 8. التقسيم الزمني وتطهير الفجوات (Chronological Splitting & Purging)
نقوم بتقسيم البيانات زمنياً بترتيب تصاعدي صارم:
- **مجموعة التدريب (Train):** 70%
- **مجموعة التحقق (Validation):** 15%
- **مجموعة الاختبار خارج العينة (Test):** 15%
مع تطبيق **فترة التطهير (Purging Gap)** بين المجموعات لضمان عدم تداخل نوافذ العوائد المتعددة.


In [ ]:
splitter = ChronologicalSplitter(
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    purge_bars=forecast_horizon,
    embargo_bars=0,
)

splits = splitter.split(X, y, metadata=meta)

print(f"مجموعة التدريب: {splits.X_train.shape[0]} عينة ({splits.X_train.index[0]} إلى {splits.X_train.index[-1]})")
print(f"مجموعة التحقق:   {splits.X_val.shape[0]} عينة ({splits.X_val.index[0]} إلى {splits.X_val.index[-1]})")
print(f"مجموعة الاختبار: {splits.X_test.shape[0]} عينة ({splits.X_test.index[0]} إلى {splits.X_test.index[-1]})")


## 9. تدريب ومقارنة النماذج على مجموعة الاختبار (Model Benchmarking)
نقوم بمقارنة الخطوط المرجعية الحقيقية (ZeroReturn, LastValue) بنماذج تعلم الآلة المتقدمة:
- **Linear Regression & Ridge**
- **Random Forest & HistGradientBoosting**
- **XGBoost, LightGBM, CatBoost**


In [ ]:
trainer = PipelineTrainer()
models = trainer.instantiate_models()
evaluator = ModelEvaluator()

fitted_models = trainer.fit_models(models, splits.X_train, splits.y_train)

# تقييم النماذج على مجموعة الاختبار خارج العينة
comparison_df = evaluator.evaluate_all(fitted_models, splits.X_test, splits.y_test, splits.meta_test)

print("جدول مقارنة النماذج على مجموعة الاختبار الحقيقية (Out-of-Sample Test Set):")
display(comparison_df[["model_name", "mae", "rmse", "r2", "directional_accuracy"]].sort_values(by="rmse"))


## 10. التنبؤ المرن بأي أفق زمني ومخروط عدم اليقين (Dynamic Horizon Forecasting & Uncertainty Cone)
يتيح النظام التنبؤ بأي عدد ساعات مستقبلي يختاره المستخدم (مثلاً 17 ساعة)، مع استخراج التوقع الإجماعي ومخروط الثقة الإحصائي $\pm 1.96\sigma$، وعرض المخطط البياني المكتمل.


In [ ]:
custom_horizon = 17  # أفق التنبؤ المطلوب: 17 ساعة في المستقبل

# تدريب سريع للأفق المخصص
dyn_engineer = FeatureEngineer(primary_horizon=custom_horizon, include_patterns=True)
dyn_X, dyn_y, dyn_meta = dyn_engineer.prepare_dataset(clean_df, target_horizon=custom_horizon)

dyn_splitter = ChronologicalSplitter(train_ratio=0.80, val_ratio=0.10, test_ratio=0.10, purge_bars=custom_horizon)
dyn_splits = dyn_splitter.split(dyn_X, dyn_y, metadata=dyn_meta)

fitted_models_dyn = trainer.fit_models(models, dyn_splits.X_train, dyn_splits.y_train)

# حساب التنبؤ اللحظي الحالي
latest_features = dyn_engineer.compute_features(clean_df).iloc[[-1]]
model_preds = {}
for name, model in fitted_models_dyn.items():
    if hasattr(model, "predict"):
        model_preds[name] = float(model.predict(latest_features)[0])

pred_returns = [r for n, r in model_preds.items() if "Baseline" not in n]
ensemble_return = float(np.mean(pred_returns))
current_price = float(clean_df["close"].iloc[-1])
target_price = current_price * (1.0 + ensemble_return)
volatility_h = float(clean_df["log_return"].tail(48).std() * np.sqrt(custom_horizon))
lower_price = target_price * (1.0 - 1.96 * volatility_h)
upper_price = target_price * (1.0 + 1.96 * volatility_h)

print(f"السعر اللحظي الحالي: ${current_price:,.2f}")
print(f"السعر المتوقع بعد {custom_horizon} ساعة: ${target_price:,.2f} ({ensemble_return*100:+.2f}%)")
print(f"مخروط عدم اليقين (95% CI): ${lower_price:,.2f} إلى ${upper_price:,.2f}")

# توليد وعرض الرسم البياني
visualizer = AdvancedChartVisualizer()
fig = visualizer.render_matplotlib_projection(
    df=clean_df,
    symbol=symbol,
    horizon=custom_horizon,
    predicted_price=target_price,
    lower_bound=lower_price,
    upper_bound=upper_price,
)
plt.show()


## 11. الخلاصة والاستنتاجات النهائية (Summary & Conclusions)

### أبرز ما أثبته هذا المشروع:
1. **الخلو التام من تسريب البيانات:** لا يمكن لأي نموذج تحقيق دقة وهمية نظراً لفصل الخصائص سببياً واستخدام فجوة التطهير واختبار التشويش المستقبلي.
2. **أهمية الخطوط المرجعية:** في الأسواق عالية الكفاءة مثل الكريبتو، يعتبر التفوق على الخط المرجعي الصفري (`ZeroReturn`) تحدياً حقيقياً ينجح فيه نموذج `CatBoost` و `Ridge`.
3. **المرونة الزمنية التامة:** إمكانية تخصيص أي أفق زمني ($h$) مع التكيف التلقائي للخصائص ومقاييس التقلب ومخاريط عدم اليقين.
4. **تكامل النماذج الفنية:** دمج إشارات التحليل الفني الكلاسيكي مع خوارزميات التعلم الآلي الحديثة يعزز قوة اتخاذ القرار وإدارة المخاطر.

---
> **إخلاء مسؤولية:** هذا النظام تم تطويره للأغراض البحثية والهندسية، ولا يمثل أي نصيحة استثمارية أو دعوة للشراء أو البيع.
